# preparation et exploration des donnees

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

In [ ]:
import pandas as pd

commandes = pd.read_excel('data.xlsx')

commandes

## identifier les principales caracteristiques des donnees

In [ ]:
commandes.info()
commandes.describe()
commandes.shape
commandes.ndim

In [ ]:
commandes.describe()

In [ ]:
# # commandes.isna().sum()
commandes.duplicated().sum()
# (commandes['Quantity'] <= 0).sum()
# (commandes['UnitPrice'] <= 0).sum()

## realiser le nettoyage necessaire 

In [ ]:
commandes = commandes.dropna(subset=['CustomerID'])
commandes = commandes[~commandes['InvoiceNo'].astype(str).str.startswith("C")]
commandes['CustomerID'] = commandes['CustomerID'].astype(int)
commandes = commandes.drop(columns=['Description', 'StockCode', 'Country'])
commandes['InvoiceDate'] = pd.to_datetime(commandes['InvoiceDate'])
commandes = commandes[commandes['Quantity'] > 0]
commandes = commandes[commandes['UnitPrice'] > 0]
commandes['TotalPrice'] = commandes['Quantity'] * commandes['UnitPrice']

# construction des indicateurs RFM

In [ ]:
date_reference = commandes['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = commandes.groupby('CustomerID').agg(
    Recence=('InvoiceDate', lambda x: (date_reference - x.max()).days),
    Frequence=('InvoiceNo', 'nunique'),
    Montant=('TotalPrice', 'sum')
).reset_index()



In [ ]:
# -------------------------
# 1. Create transformed RFM
# -------------------------

rfm_transformed = rfm.copy()

# Recence: make recent customers have larger values
rfm_transformed['Recence'] = -rfm_transformed['Recence']

# Log-transform skewed variables
rfm_transformed['Frequence'] = np.log1p(
    rfm_transformed['Frequence']
)

rfm_transformed['Montant'] = np.log1p(
    rfm_transformed['Montant']
)

In [ ]:
plt.hist(rfm_transformed['Montant'],bins=30)
plt.show()